# 🌳 Decision Trees
**ISLP Chapter 8 · Pattern Recognition for the Rest of Us**

> Decision trees split data by asking a sequence of yes/no questions. They're one of the most interpretable ML models — you can literally draw them out and explain them to anyone.

### What you'll learn
- How trees split using Gini impurity and RSS
- Growing and pruning trees to avoid overfitting
- Classification trees vs regression trees
- Tree visualization and interpretation
- Why trees are the building block for Random Forests and Boosting

### Dataset: Carseats (sales prediction) + Heart disease (classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Carseats

In [ ]:
import pandas as pd
carseats = pd.read_csv(f'{DATA_DIR}/Carseats.csv')
carseats = pd.get_dummies(carseats, drop_first=True, dtype=float)
X_reg = carseats.drop('Sales', axis=1)
y_reg = carseats['Sales']
print(f"Carseats: {X_reg.shape}  |  Predicting: Sales")

### Load Dataset: Heart

In [ ]:
import pandas as pd
heart = pd.read_csv(f'{DATA_DIR}/Heart.csv').dropna()
heart = pd.get_dummies(heart, drop_first=True, dtype=float)
target_col = [c for c in heart.columns if 'AHD' in c or c == heart.columns[-1]][0]
X = heart.drop(target_col, axis=1)
y = heart[target_col].astype(int)
print(f"Heart: {X.shape}  |  Disease rate: {y.mean():.1%}")

## 🌳 Part 1 — How Trees Split

**For regression trees:** At each node, find the feature j and threshold s that minimize RSS:
```
RSS = Σ(yᵢ - ȳ_left)² + Σ(yᵢ - ȳ_right)²
```

**For classification trees:** Minimize **Gini impurity** at each node:
```
Gini = 1 - Σₖ p̂ₖ²
```
where p̂ₖ is the proportion of class k observations in the node.
A node with all one class has Gini=0 (pure). A 50/50 split has Gini=0.5 (most impure).

In [ ]:
# Visualize Gini impurity
p = np.linspace(0.001, 0.999, 200)
gini = 1 - p**2 - (1-p)**2
entropy = -p*np.log2(p) - (1-p)*np.log2(1-p)
misclass = 1 - np.maximum(p, 1-p)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p, gini,     color='#1e5fa8', lw=2, label='Gini impurity')
ax.plot(p, entropy/2, color='#e85d20', lw=2, ls='--', label='Entropy/2')
ax.plot(p, misclass, color='#888',    lw=1.5, ls=':', label='Misclassification rate')
ax.set_xlabel('Proportion of class 1 in node')
ax.set_ylabel('Impurity measure')
ax.set_title('Node Impurity Measures — all maximized at 50/50 split')
ax.legend()
ax.axvline(0.5, color='black', lw=0.8, ls='--', alpha=0.4)
plt.tight_layout()
plt.show()
print("📌 Trees try to create pure nodes (low Gini). A pure node = all observations are one class.")

In [ ]:
# Grow and visualize a classification tree
X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

tree_clf = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
tree_clf.fit(X_tr, y_tr)

print(f"Training accuracy: {tree_clf.score(X_tr, y_tr):.3f}")
print(f"Test accuracy:     {tree_clf.score(X_te, y_te):.3f}")
print(f"Number of leaves:  {tree_clf.get_n_leaves()}")

fig, ax = plt.subplots(figsize=(20, 7))
plot_tree(tree_clf, feature_names=X_clf.columns, class_names=['No Disease','Disease'],
          filled=True, rounded=True, fontsize=8, ax=ax,
          impurity=True, proportion=False)
ax.set_title('Classification Tree (max_depth=4) — Heart Disease Prediction', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# The overfitting problem — training vs test accuracy by tree depth
depths = range(1, 20)
train_acc, test_acc = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_tr, y_tr)
    train_acc.append(t.score(X_tr, y_tr))
    test_acc.append(t.score(X_te, y_te))

best_depth = depths[test_acc.index(max(test_acc))]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(depths), train_acc, 'o-', color='#1e5fa8', lw=2, label='Training accuracy')
ax.plot(list(depths), test_acc,  's-', color='#e85d20', lw=2, label='Test accuracy')
ax.axvline(best_depth, color='#1a7a45', lw=1.5, ls='--', label=f'Best depth = {best_depth}')
ax.set_xlabel('Max Tree Depth')
ax.set_ylabel('Accuracy')
ax.set_title('Overfitting: Deep Trees Memorize Training Data')
ax.legend()
plt.tight_layout()
plt.show()
print(f"\n📌 Training accuracy → 100% as depth increases (memorizes every point)")
print(f"   Test accuracy peaks at depth {best_depth} then falls — overfitting begins")

In [ ]:
# Regression tree on Carseats
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)
tree_reg = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_reg.fit(X_tr_r, y_tr_r)

train_mse = mean_squared_error(y_tr_r, tree_reg.predict(X_tr_r))
test_mse  = mean_squared_error(y_te_r, tree_reg.predict(X_te_r))

print(f"Regression Tree (max_depth=4) — Carseats Sales")
print(f"Training RMSE: {np.sqrt(train_mse):.3f}")
print(f"Test RMSE:     {np.sqrt(test_mse):.3f}")

# Feature importance
imp = pd.Series(tree_reg.feature_importances_, index=X_reg.columns).sort_values(ascending=False).head(8)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(imp.index, imp.values, color='#1e5fa8', edgecolor='white')
ax.set_ylabel('Feature Importance')
ax.set_title('Regression Tree — Feature Importance for Carseats Sales')
ax.set_xticklabels(imp.index, rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# @title 📝 Quiz — Decision Trees
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is Gini impurity and what value indicates a pure node?
# @markdown **Q2:** Why do very deep trees overfit?
# @markdown **Q3:** What is the difference between pruning and setting max_depth?
# @markdown **Q4:** Name one advantage and one disadvantage of decision trees vs linear models
# @markdown **Q5:** Why are decision trees the building block for Random Forests and Boosting?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Decision Trees"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username below so your instructor can track your submission, then click ▶ Run.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    qa       = "\n".join(f"Q{k.replace('q','')}: {v}" for k,v in answered.items())
    prompt   = (f"You are a TA grading quiz answers for \"{nb_name}\" in a data science "
                f"course (Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
                f"Student answers ({len(answered)}/{n_total}):\n{qa or '(none)'}\n\n"
                f"Grade conceptual understanding. Accept reasonable paraphrases. "
                f"Be encouraging and specific. Reply ONLY in this JSON (no markdown):\n"
                f"{{\"quiz_score\":<int 0-{n_total}>,"
                f"\"grade\":\"<Excellent|Good|Needs Review|Incomplete>\","
                f"\"feedback\":\"<2-3 sentences>\","
                f"\"tip\":\"<one takeaway>\"}}") 
    try:
        import google.generativeai as genai
        import google.auth, google.auth.transport.requests
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
        return r.text, "Gemini via Colab"
    except Exception:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
            return r.text, "Gemini via key"
    except Exception:
        pass
    return None, None

def _fallback(answers_dict):
    n = len(answers_dict)
    done = [v for v in answers_dict.values() if str(v).strip()]
    nd, avg = len(done), sum(len(str(v)) for v in done)/max(len(done),1)
    if nd == 0: return {"quiz_score":0,"grade":"Incomplete","feedback":"No answers — fill in the quiz above.","tip":"Each question needs 1-2 sentences."}
    if avg < 15: return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered but very brief. Explain your reasoning.","tip":"Aim for 1-2 sentences per answer."}
    if nd == n:  return {"quiz_score":n,"grade":"Good","feedback":f"All {n} questions answered.","tip":"Review any concepts that felt unclear."}
    return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered. Complete the remaining {n-nd}.","tip":"Answer all questions for full credit."}

def _show(result, username, nb_name, source):
    C = {"Excellent":"\033[92m","Good":"\033[94m","Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R = "\033[0m"; g = result.get("grade","?")
    n = len(answers); qs = result.get("quiz_score",0)
    print("\n"+"\u2550"*56)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*56)
    print(f"  Student : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade   : {C.get(g,'')} {g} {R}")
    print(f"  Score   : {qs}/{n}  [{'\u2588'*qs+'\u2591'*(n-qs)}]")
    print()
    [print(f"  {l}") for l in textwrap.wrap(result.get("feedback",""),54)]
    tip = result.get("tip","")
    if tip:
        print()
        [print(f"  \U0001f4a1 {l}") for l in textwrap.wrap(tip,52)]
    print("\u2550"*56+"\n")

if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd = sum(1 for v in answers.values() if str(v).strip())
    print(f"  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    raw, src = _grade(answers, _NB_TITLE)
    if raw:
        try:
            result = json.loads(_re.sub(r"```(?:json)?\s*|```","",raw).strip())
        except Exception:
            result = {"quiz_score":nd,"grade":"Good" if nd==len(answers) else "Needs Review","feedback":raw[:400],"tip":""}
    else:
        print("  (Gemini unavailable \u2014 using completion-based feedback)\n")
        result = _fallback(answers)
        src = None
    _show(result, GITHUB_USERNAME.strip(), _NB_TITLE, src)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")
